# Lab 09 · Association Rules & Frequent Itemsets
**Advanced Data Mining — M.Sc. Data Science** · Dr. Ali Louati · Prince Sattam Bin Abdulaziz University
Companion to **Chapter 9**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alilouati/advanced-data-mining/blob/main/labs/Lab09_Association_Rules.ipynb)

**In this lab you will:**
- count itemset supports by hand
- mine frequent itemsets with Apriori and FP-Growth
- generate rules and rank them by confidence and lift
- visualise rules as a scatter plot, heatmap and network

> **How to use:** run the cells from top to bottom (`Shift + Enter`). Cells marked ✏️ are for you to modify.
> Every step works with the built-in data set; set `USE_MY_CSV = True` to try your own CSV file.

In [ ]:
# mlxtend provides Apriori / FP-Growth (pre-installed on Colab; this line installs it elsewhere)
!pip install -q mlxtend

In [ ]:
# Setup: libraries used in this lab (all pre-installed on Google Colab)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (8, 4.5)
np.random.seed(42)

## 1 · Shopping baskets

In [ ]:
baskets = [
    ["bread", "milk", "dates"], ["bread", "coffee", "dates"], ["milk", "coffee", "cardamom", "dates"],
    ["bread", "milk", "eggs"], ["bread", "milk", "dates", "eggs"], ["coffee", "cardamom", "dates"],
    ["bread", "eggs", "cheese"], ["milk", "cheese", "bread"], ["coffee", "cardamom"],
    ["bread", "milk", "cheese", "eggs"], ["dates", "coffee", "milk"], ["bread", "dates"],
    ["coffee", "cardamom", "dates", "milk"], ["eggs", "cheese"], ["bread", "milk", "dates", "coffee"],
    ["cardamom", "coffee", "bread"], ["milk", "eggs"], ["bread", "cheese", "dates"],
    ["coffee", "dates", "cardamom"], ["bread", "milk"],
]
print(len(baskets), "transactions")

In [ ]:
# ✏️ Optional: your own transactions — a CSV with one row per (transaction_id, item)
USE_MY_CSV = False
if USE_MY_CSV:
    from google.colab import files
    up = files.upload()
    raw = pd.read_csv(next(iter(up)))
    baskets = raw.groupby(raw.columns[0])[raw.columns[1]].apply(list).tolist()
    print(len(baskets), "transactions loaded")

## 2 · Support by hand

In [ ]:
from itertools import combinations
from collections import Counter

n = len(baskets)
item_counts = Counter(item for b in baskets for item in set(b))
pair_counts = Counter(pair for b in baskets for pair in combinations(sorted(set(b)), 2))

support_items = pd.Series({k: v / n for k, v in item_counts.items()}).sort_values()
support_items.plot(kind="barh", color="teal"); plt.axvline(0.3, ls="--", color="red")
plt.title("Item support (red line = minsup 0.3)"); plt.show()
pd.Series({f"{a} & {b}": v / n for (a, b), v in pair_counts.most_common(8)}, name="support")

In [ ]:
def confidence(X, Y):
    """conf(X -> Y) = support(X and Y) / support(X)"""
    both = sum(1 for b in baskets if X in b and Y in b)
    return both / sum(1 for b in baskets if X in b)

def lift(X, Y):
    return confidence(X, Y) / (sum(1 for b in baskets if Y in b) / n)

print(f"coffee -> cardamom : conf = {confidence('coffee', 'cardamom'):.2f}, lift = {lift('coffee', 'cardamom'):.2f}")
print(f"bread  -> milk     : conf = {confidence('bread', 'milk'):.2f}, lift = {lift('bread', 'milk'):.2f}")

## 3 · Apriori and FP-Growth with mlxtend

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

te = TransactionEncoder()
onehot = pd.DataFrame(te.fit(baskets).transform(baskets), columns=te.columns_)
display(onehot.head())

MIN_SUPPORT = 0.15                                   # ✏️ try 0.1, 0.2 …
freq = fpgrowth(onehot, min_support=MIN_SUPPORT, use_colnames=True)
freq["size"] = freq["itemsets"].apply(len)
print(len(freq), "frequent itemsets")
freq.sort_values("support", ascending=False).head(10)

In [ ]:
# Apriori gives exactly the same itemsets (it is just slower on big data)
same = set(map(frozenset, apriori(onehot, min_support=MIN_SUPPORT, use_colnames=True)["itemsets"])) == set(map(frozenset, freq["itemsets"]))
print("Apriori == FP-Growth:", same)

## 4 · Association rules ranked by lift

In [ ]:
MIN_CONF = 0.6                                       # ✏️ minimum confidence
try:    # newer mlxtend versions ask for the number of transactions
    rules = association_rules(freq, num_itemsets=len(onehot), metric="confidence", min_threshold=MIN_CONF)
except TypeError:
    rules = association_rules(freq, metric="confidence", min_threshold=MIN_CONF)
rules = rules.sort_values("lift", ascending=False)
rules["rule"] = rules["antecedents"].apply(lambda s: ", ".join(s)) + "  ->  " + rules["consequents"].apply(lambda s: ", ".join(s))
rules[["rule", "support", "confidence", "lift"]].round(2).head(10)

In [ ]:
sc = plt.scatter(rules["support"], rules["confidence"], c=rules["lift"], s=200 * rules["lift"], cmap="viridis", edgecolor="k")
plt.colorbar(sc, label="lift"); plt.xlabel("support"); plt.ylabel("confidence")
plt.title("Rules: support vs. confidence (colour & size = lift)"); plt.show()

In [ ]:
import networkx as nx
G = nx.DiGraph()
for _, r in rules.head(12).iterrows():
    for a in r["antecedents"]:
        for c in r["consequents"]:
            G.add_edge(a, c, weight=r["lift"])
pos = nx.circular_layout(G)
nx.draw_networkx(G, pos, node_color="lightblue", node_size=1800, font_size=9,
                 width=[1.5 * G[u][v]["weight"] for u, v in G.edges()], edge_color="gray", arrowsize=15)
plt.title("Top rules as a network (thicker = higher lift)"); plt.axis("off"); plt.show()

## ✏️ Exercises
1. Lower `MIN_SUPPORT` to 0.1. How many itemsets and rules appear now?
2. Find a rule with high confidence but lift < 1. Why is it misleading?
3. Add a new item (e.g. `"tea"`) to several baskets and observe the rules.
4. Run the lab on your own transaction CSV.